In [1]:
# [1] 설치 (필요 시 1회)
!pip install -q lightgbm xgboost


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
# [2] 기본 설정 & 임포트
import os, gc, warnings, random, math
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

import lightgbm as lgb
import xgboost as xgb

os.environ["TQDM_NOTEBOOK"] = "0"
from tqdm import tqdm as _tqdm

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED); random.seed(SEED)

DATA_DIR = "/mnt/elice/dataset"

# ===== 속도/안정 스위치 =====
FAST_MODE      = False     # ★ 최종 제출: False
THR_MODE       = "fold"    # 'oof' | 'fold' | 'rate' (fold 권장)
RATE_BUF       = 1.03      # 'rate'일 때 pos비율 × 버퍼
CALIB_ISO_2ND  = False     # 2단계 확률 Isotonic 보정 (원하면 True)
BLEND_SPACE    = "auto"    # 'prob' | 'logit' | 'auto'

# 1단계(행단계) 라운드/가중
ROW_POS_GAIN   = 1.5
ROW_LGB_ROUNDS = 2200
ROW_LGB_ES     = 240

# 2단계(시리얼) 라운드
LGB2_ROUNDS = 3600
LGB2_ES     = 240
XGB2_ROUNDS = 3400
XGB2_ES     = 240
HGB2_ITERS  = 800

# 1/2단계 CV
N_SPLITS_ROW = 5
N_SPLITS_2ND = 5

In [3]:
# [3] 데이터 적재 (train-only 의사결정 준수)
train   = pd.read_csv(f"{DATA_DIR}/train.csv")
train_y = pd.read_csv(f"{DATA_DIR}/train_y.csv")   # Serial Number 단위 Y
test    = pd.read_csv(f"{DATA_DIR}/test_x.csv")

# TIMESTAMP는 반드시 datetime
for df in (train, test):
    df["TIMESTAMP"] = pd.to_datetime(
        df["TIMESTAMP"], format="%Y.%m.%d %I:%M:%S %p", errors="coerce"
    )

# X1은 문자열 → 2단계에서 target encoding으로만 사용
feature_cols = [c for c in train.columns if c.startswith("X") and c!="X1"]
len(feature_cols), feature_cols[:5], train["TIMESTAMP"].dtype

(17, ['X2', 'X3', 'X4', 'X5', 'X6'], dtype('<M8[ns]'))

In [4]:
# [4] 유틸 (통계/추세/임계값)
from scipy.stats import skew, kurtosis, linregress, kendalltau

def _safe_last(a, k=1):
    if len(a)==0: return 0.0
    k=min(k,len(a)); return float(np.mean(a[-k:]))

def _safe_first(a, k=1):
    if len(a)==0: return 0.0
    k=min(k,len(a)); return float(np.mean(a[:k]))

def _slope_r2(idx01, a):
    if len(a)>=3:
        slope, _, r, _, _ = linregress(idx01, a)
        return float(slope), float(r**2)
    return 0.0, 0.0

def _diff_stats(a):
    if len(a)<2: return (0.0,)*4
    d = np.diff(a)
    return float(np.mean(np.abs(d))), float(np.max(np.abs(d))), float(np.mean(d>0)), float(np.mean(d<0))

def _resid_std(idx, a):
    if len(a)<3: return 0.0
    slope, intercept = np.polyfit(idx, a, 1)
    resid = a - (slope*idx + intercept)
    return float(np.std(resid))

def _pos_of_extreme(a):
    if len(a)==0: return 0.0, 0.0
    imax, imin, n = int(np.argmax(a)), int(np.argmin(a)), len(a)
    return float((imax+1)/n), float((imin+1)/n)

def _roll_last(a, win, fn=np.std):
    if len(a)<win: return 0.0
    return float(fn(a[-win:]))

def _iqr_outlier_ratio(a):
    if len(a)==0: return 0.0
    q25, q75 = np.percentile(a, [25, 75]); iqr = q75-q25
    if iqr==0: return 0.0
    lo, hi = q25-1.5*iqr, q75+1.5*iqr
    return float(np.mean((a<lo)|(a>hi)))

def _kendall_tau(a):
    if len(a)<3: return 0.0
    try:
        tau,_ = kendalltau(np.arange(len(a)), a)
        return float(0.0 if np.isnan(tau) else tau)
    except:
        return 0.0

def _autocorr_lag1(a):
    if len(a)<2: return 0.0
    a = a - np.mean(a)
    num = np.sum(a[1:]*a[:-1]); den = np.sum(a*a)+1e-9
    return float(num/den)

def _run_lengths(a):
    if len(a)<3: return 1.0, 1.0
    d = np.sign(np.diff(a))
    inc = (d>0).astype(int); dec=(d<0).astype(int)
    def _avg_run(x):
        runs=[]; cur=0
        for v in x:
            if v==1: cur+=1
            else:
                if cur>0: runs.append(cur); cur=0
        if cur>0: runs.append(cur)
        return float(np.mean(runs)) if runs else 1.0
    return _avg_run(inc), _avg_run(dec)

def _cusum_count(d, k=3.5):
    if len(d)==0: return 0
    ad = np.abs(d)
    med = np.median(ad); mad = np.median(np.abs(ad-med))
    thr = med + k*(1.4826*mad + 1e-9)
    return int(np.sum(ad>thr))

def _quantiles(a, qs=(0.75,0.90,0.95)):
    if len(a)==0: return {f"q{int(q*100)}":0.0 for q in qs}
    vals = np.quantile(a, qs)
    return {f"q{int(q*100)}": float(v) for q,v in zip(qs, vals)}

def _rank_norm(a):
    r = pd.Series(a).rank(method="average").values
    return r / (len(a)+1e-9)

def _logit(p):
    p = np.clip(p, 1e-6, 1-1e-6); return np.log(p/(1-p))

def _ilogit(z):
    return 1.0/(1.0+np.exp(-z))

# 공용: F1(macro) 최대 임계값
def _best_f1_thr(y_true, p, average="macro"):
    best_f1, best_thr = -1.0, 0.5
    for thr in np.linspace(0.05, 0.95, 181):
        f1 = f1_score(y_true, (p>thr).astype(int), average=average)
        if f1 > best_f1: best_f1, best_thr = f1, thr
    return best_f1, best_thr

In [5]:
# [5] (train-only) 변동 큰 원천 컬럼 Top‑K 선택  ← 규정 안전
def select_high_var_cols_train_only(train, feature_cols, top_k=6):
    variances=[]
    for c in feature_cols:
        arr = pd.to_numeric(train[c], errors="coerce").astype(float).values
        arr = pd.Series(arr).interpolate("linear", limit_direction="both").fillna(0.0).values
        variances.append((c, float(np.var(arr))))
    variances.sort(key=lambda x: -x[1])
    return [c for c,_ in variances[:top_k]]

TOPK_PAIR = 6
topk_cols = select_high_var_cols_train_only(train, feature_cols, top_k=TOPK_PAIR)
topk_cols

['X13', 'X8', 'X9', 'X10', 'X17', 'X18']

In [6]:
# [6] (A) 대형 시리얼 집계 피처 (last/mean/std/추세/런/이상치/최근성/ratio 등)
def make_serial_features(df, feature_cols, topk_cols):
    feats=[]
    for sn, g in df.groupby("Serial Number"):
        g = g.sort_values("TIMESTAMP")
        row={"Serial Number": sn, "len": len(g)}

        # 시간 축/간격
        if len(g)>=2 and pd.notnull(g["TIMESTAMP"]).all():
            span_s = (g["TIMESTAMP"].iloc[-1] - g["TIMESTAMP"].iloc[0]).total_seconds()
            row["span_days"] = float(span_s/86400.0)
            dt = g["TIMESTAMP"].diff().dt.total_seconds().fillna(0).values
            row["dt_mean_s"]=float(np.mean(dt)); row["dt_std_s"]=float(np.std(dt))
            row["samples_per_day"]= (len(g)/max(row["span_days"], 1e-6))
            t = g["TIMESTAMP"].astype("int64").values / 1e9
            t = (t - t[0]) / max((t[-1]-t[0]), 1e-9)
        else:
            row["span_days"]=0.0; row["dt_mean_s"]=0.0; row["dt_std_s"]=0.0
            row["samples_per_day"]=float(len(g))
            t = np.linspace(0.0,1.0,len(g)) if len(g)>0 else np.array([0.0])

        idx = np.arange(len(g)).astype(float)

        for col in feature_cols:
            ser = pd.to_numeric(g[col], errors="coerce").astype(float)
            nmiss = ser.isna().sum()
            arr = ser.interpolate("linear", limit_direction="both").fillna(0.0).values

            m=float(np.mean(arr)); s=float(np.std(arr, ddof=0))
            mn=float(np.min(arr));  mx=float(np.max(arr))
            med=float(np.median(arr))
            q25, q75 = np.percentile(arr, [25, 75]); iqr=float(q75-q25); rng=float(mx-mn)

            row[f"{col}_nmiss"]=int(nmiss)
            row[f"{col}_last"]=float(arr[-1]) if len(arr) else 0.0
            row[f"{col}_mean"]=m; row[f"{col}_std"]=s
            row[f"{col}_min"]=mn; row[f"{col}_max"]=mx
            row[f"{col}_median"]=med; row[f"{col}_q25"]=float(q25); row[f"{col}_q75"]=float(q75)
            row[f"{col}_iqr"]= iqr; row[f"{col}_range"]=rng
            row[f"{col}_skew"]= float(skew(arr)) if len(arr)>2 else 0.0
            row[f"{col}_kurt"]= float(kurtosis(arr)) if len(arr)>3 else 0.0

            row[f"{col}_diff"]= float(arr[-1]-arr[0]) if len(arr)>0 else 0.0
            row[f"{col}_last_minus_mean"]= float(arr[-1]-m)
            row[f"{col}_last_minus_median"]= float(arr[-1]-med)
            row[f"{col}_cv"]= float(s/(m+1e-6))
            slope_t, r2_t = _slope_r2(t, arr)
            row[f"{col}_slope_t"]=slope_t; row[f"{col}_r2_t"]=r2_t
            row[f"{col}_resid_std"]= _resid_std(idx, arr)
            row[f"{col}_kendall_tau"]= _kendall_tau(arr)

            row[f"{col}_last3_mean"] = _safe_last(arr,3)
            row[f"{col}_last5_mean"] = _safe_last(arr,5)
            row[f"{col}_last10_mean"]= _safe_last(arr,10)
            row[f"{col}_first5_mean"]= _safe_first(arr,5)

            row[f"{col}_roll5_std_last"]   = _roll_last(arr,5, np.std)
            row[f"{col}_roll10_std_last"]  = _roll_last(arr,10,np.std)
            row[f"{col}_roll10_mean_last"] = _roll_last(arr,10,np.mean)

            row[f"{col}_acf1"]= _autocorr_lag1(arr)
            inc_run, dec_run = _run_lengths(arr)
            row[f"{col}_inc_run"]=inc_run; row[f"{col}_dec_run"]=dec_run

            row[f"{col}_last_z"]= float((arr[-1]-m)/(s+1e-6))
            pmax, pmin = _pos_of_extreme(arr)
            row[f"{col}_pos_of_max"]=pmax; row[f"{col}_pos_of_min"]=pmin

            row[f"{col}_iqr_out_ratio"]= _iqr_outlier_ratio(arr)
            row[f"{col}_pct_above_q75"]= float(np.mean(arr>q75)) if iqr>0 else 0.0
            row[f"{col}_pct_below_q25"]= float(np.mean(arr<q25)) if iqr>0 else 0.0

            madiff,maxdiff,p_up,p_down = _diff_stats(arr)
            row[f"{col}_madiff"]=madiff; row[f"{col}_maxdiff"]=maxdiff
            row[f"{col}_p_up"]=p_up; row[f"{col}_p_down"]=p_down

            half=max(len(arr)//2,1)
            row[f"{col}_first_half_mean"]= float(np.mean(arr[:half]))
            row[f"{col}_last_half_mean"] = float(np.mean(arr[half:]))
            row[f"{col}_half_gap"]= float(np.mean(arr[half:]) - np.mean(arr[:half]))

            if len(arr)>=3:
                alpha=0.3
                ewm = pd.Series(arr).ewm(alpha=alpha, adjust=False).mean().values
                row[f"{col}_ewm_last"]= float(ewm[-1])
                row[f"{col}_ewm_gap"]= float(ewm[-1]-m)
                slope_e,r2_e = _slope_r2(np.linspace(0,1,len(ewm)), ewm)
                row[f"{col}_ewm_slope"]=slope_e; row[f"{col}_ewm_r2"]=r2_e
            else:
                row[f"{col}_ewm_last"]= arr[-1] if len(arr) else 0.0
                row[f"{col}_ewm_gap"]= (arr[-1]-m) if len(arr) else 0.0
                row[f"{col}_ewm_slope"]=0.0; row[f"{col}_ewm_r2"]=0.0

        # 상호작용 (Top‑K 마지막값 ratio/diff)
        for i in range(len(topk_cols)):
            for j in range(i+1, len(topk_cols)):
                a = pd.to_numeric(g[topk_cols[i]], errors="coerce").astype(float).interpolate("linear", limit_direction="both").fillna(0.0).values
                b = pd.to_numeric(g[topk_cols[j]], errors="coerce").astype(float).interpolate("linear", limit_direction="both").fillna(0.0).values
                if len(a)==0 or len(b)==0:
                    row[f"{topk_cols[i]}_div_{topk_cols[j]}_last"]=0.0
                    row[f"{topk_cols[i]}_minus_{topk_cols[j]}_last"]=0.0
                else:
                    row[f"{topk_cols[i]}_div_{topk_cols[j]}_last"]= float(a[-1]/(b[-1]+1e-6))
                    row[f"{topk_cols[i]}_minus_{topk_cols[j]}_last"]= float(a[-1]-b[-1])

        feats.append(row)

    out = pd.DataFrame(feats).set_index("Serial Number")
    out = out.replace([np.inf,-np.inf], np.nan).fillna(0.0)
    return out.astype("float32")

X_train_big = make_serial_features(train, feature_cols, topk_cols)
X_test_big  = make_serial_features(test , feature_cols, topk_cols)
X_train_big.shape, X_test_big.shape

((8272, 851), (2069, 851))

In [7]:
# [7] (B) 행 단계: train-only 고정 rank로 파생 + GroupCV LGB → p_row
def build_row_features(df, feature_cols, topk_diff=6, fixed_var_cols=None):
    df = df.copy()
    # 숫자화
    Xnum = df[feature_cols].apply(pd.to_numeric, errors="coerce").astype(float)
    for c in feature_cols: df[c] = Xnum[c]
    df.sort_values(["Serial Number","TIMESTAMP"], inplace=True)
    grp = df.groupby("Serial Number")

    # 고정 순서(핵심: 테스트/트레인 일관)
    df["ORD"] = grp.cumcount().astype("int32")

    # 시간 파생
    dt_sec   = grp["TIMESTAMP"].diff().dt.total_seconds().fillna(0).clip(lower=0).astype("float32")
    span_sec = (grp["TIMESTAMP"].transform("max") - grp["TIMESTAMP"].transform("min")).dt.total_seconds().replace(0,1e-6)
    pos01    = (grp.cumcount() / (grp["TIMESTAMP"].transform("size")-1).replace(0,1)).astype(float)
    to_last  = (grp["TIMESTAMP"].transform("max") - df["TIMESTAMP"]).dt.total_seconds().astype(float)

    hh = df["TIMESTAMP"].dt.hour.fillna(0).astype(float); mn = df["TIMESTAMP"].dt.minute.fillna(0).astype(float)
    frac = (hh + mn/60.0)/24.0
    df["dt_log1p"]=np.log1p(dt_sec)
    df["pos01"]=pos01; df["to_last01"]=(to_last/(span_sec+1e-6)).astype(float)
    df["hr_sin"]=np.sin(2*np.pi*frac).astype(float); df["hr_cos"]=np.cos(2*np.pi*frac).astype(float)

    # train-only로 고정한 var 랭크 사용
    var_rank = list(fixed_var_cols)[:topk_diff]

    for c in var_rank:
        g = grp[c]
        df[f"{c}_diff1"]= g.diff().fillna(0).astype("float32")
        df[f"{c}_r3"]   = g.transform(lambda s: s.rolling(3, min_periods=1).mean()).astype("float32")
        mu=g.transform("mean"); sd=g.transform("std").replace(0,1e-6)
        df[f"{c}_z"]    = ((df[c]-mu)/sd).astype("float32")

    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].replace([np.inf,-np.inf], np.nan).fillna(0.0)

    row_feat_cols = (feature_cols
                     + [f"{c}_diff1" for c in var_rank]
                     + [f"{c}_r3"    for c in var_rank]
                     + [f"{c}_z"     for c in var_rank]
                     + ["dt_log1p","pos01","to_last01","hr_sin","hr_cos"])
    return df[["Serial Number","TIMESTAMP","ORD"]+row_feat_cols], row_feat_cols

# train 기준 topK 고정 (train-only)
TOPK_DIFF = 6
train_num_for_rank = train[feature_cols].apply(pd.to_numeric, errors="coerce").astype(float)
fixed_var_cols = train_num_for_rank.var().sort_values(ascending=False).index.tolist()[:TOPK_DIFF]

row_train, row_cols = build_row_features(train, feature_cols, topk_diff=TOPK_DIFF, fixed_var_cols=fixed_var_cols)
row_test , _        = build_row_features(test , feature_cols, topk_diff=TOPK_DIFF, fixed_var_cols=fixed_var_cols)

def train_row_lgb(train_rows, y_serial_map, test_rows, row_feat_cols,
                  num_rounds=ROW_LGB_ROUNDS, es_rounds=ROW_LGB_ES, desc="[ROW-LGB]"):
    tr=train_rows.copy(); te=test_rows.copy()
    tr["Y"] = tr["Serial Number"].map(y_serial_map).astype(int)

    # 시리얼 균형 가중: 그룹 총합 1, 양성 배가
    n_per_sn = tr.groupby("Serial Number")["Y"].transform("size")
    base_w = 1.0 / n_per_sn
    pos_sn = set(y_serial_map[y_serial_map==1].index)
    tr["w"] = base_w * tr["Serial Number"].apply(lambda s: ROW_POS_GAIN if s in pos_sn else 1.0)

    X = tr[row_feat_cols].astype("float32").values
    y = tr["Y"].values.astype(int)
    w = tr["w"].values.astype(np.float32)
    Xte = te[row_feat_cols].astype("float32").values

    cv = StratifiedGroupKFold(n_splits=N_SPLITS_ROW, shuffle=True, random_state=SEED)
    oof = np.zeros(len(tr), dtype=np.float32); test_f=np.zeros((len(te), N_SPLITS_ROW), dtype=np.float32)

    p = dict(objective="binary", metric="auc",
             learning_rate=0.03 if FAST_MODE else 0.03,
             num_leaves=96, max_depth=-1,
             feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
             min_data_in_leaf=80, max_bin=255, lambda_l2=1.0, verbose=-1, seed=SEED)

    pbar=_tqdm(total=N_SPLITS_ROW, desc=desc, leave=False); k=0
    for tr_idx, va_idx in cv.split(X, y, groups=tr["Serial Number"].values):
        dtr=lgb.Dataset(X[tr_idx], label=y[tr_idx], weight=w[tr_idx], free_raw_data=False)
        dva=lgb.Dataset(X[va_idx], label=y[va_idx], weight=w[va_idx], reference=dtr, free_raw_data=False)
        m=lgb.train(p, dtr, num_boost_round=num_rounds, valid_sets=[dva],
                    callbacks=[lgb.early_stopping(es_rounds, verbose=False)])
        oof[va_idx]=m.predict(X[va_idx], num_iteration=m.best_iteration).astype(np.float32)
        test_f[:,k]=m.predict(Xte, num_iteration=m.best_iteration).astype(np.float32)
        k+=1; pbar.update(1)
        del m,dtr,dva; gc.collect()
    pbar.close()

    tr_out = tr[["Serial Number","TIMESTAMP","ORD"]].copy(); tr_out["p_row"]=oof
    te_out = te[["Serial Number","TIMESTAMP","ORD"]].copy(); te_out["p_row"]=test_f.mean(axis=1).astype(np.float32)
    return tr_out, te_out

y_map = train_y.set_index("Serial Number")["Y"].astype(int)
row_p_tr, row_p_te = train_row_lgb(row_train, y_map, row_test, row_cols, desc="[ROW-LGB]")
row_p_tr.head()

,Serial Number,TIMESTAMP,ORD,p_row
0,19,NaT,0,0.977801
1,19,NaT,1,0.947766
2,19,NaT,2,0.986397
3,19,NaT,3,0.989410
4,19,NaT,4,0.991856


In [8]:
# [8] (B') p_row 시퀀스 집계(ORD로 정렬 고정)
def aggregate_prob_features(df_rows, p_rows):
    df = df_rows[["Serial Number","ORD"]].copy().reset_index(drop=True)
    df["p"] = p_rows["p_row"].values.astype(np.float32)
    feats=[]
    for sn, g in df.groupby("Serial Number"):
        g = g.sort_values("ORD", kind="mergesort")
        arr=g["p"].values.astype(float); n=len(arr)
        idx01=np.linspace(0,1,n) if n>1 else np.array([0.0])
        row={"Serial Number":sn, "n_rows":n}
        row["p_last"]=float(arr[-1]) if n else 0.0
        row["p_mean"]=float(arr.mean() if n else 0.0)
        row["p_max"]= float(arr.max() if n else 0.0)
        row.update({f"p_{k}":v for k,v in _quantiles(arr,(0.75,0.90,0.95)).items()})
        for t in (0.5,0.7,0.8,0.9):
            row[f"p_cnt_gt_{int(t*100)}"]=int(np.sum(arr>t))
            row[f"p_rat_gt_{int(t*100)}"]=float(np.mean(arr>t))
        row["p_last3"]=_safe_last(arr,3); row["p_last5"]=_safe_last(arr,5); row["p_last10"]=_safe_last(arr,10)
        sl,r2=_slope_r2(idx01,arr); row["p_slope"]=sl; row["p_r2"]=r2
        madiff,mxdiff,p_up,p_down=_diff_stats(arr)
        row["p_madiff"]=madiff; row["p_maxdiff"]=mxdiff; row["p_p_up"]=p_up; row["p_p_down"]=p_down
        row["p_cusum_cnt"]=_cusum_count(np.diff(arr)) if n>1 else 0
        if n>=3:
            a=np.array_split(arr,3)
            row["p_s1_mean"]=float(a[0].mean()); row["p_s2_mean"]=float(a[1].mean()); row["p_s3_mean"]=float(a[2].mean())
            row["p_s3_minus_s1"]=float(a[2].mean()-a[0].mean())
        else:
            row["p_s1_mean"]=row["p_s2_mean"]=row["p_s3_mean"]=row["p_s3_minus_s1"]=0.0
        feats.append(row)
    out=pd.DataFrame(feats).set_index("Serial Number")
    return out.replace([np.inf,-np.inf], np.nan).fillna(0.0).astype("float32")

P_train = aggregate_prob_features(row_p_tr, row_p_tr)
P_test  = aggregate_prob_features(row_p_te, row_p_te)
P_train.shape, P_test.shape

((8272, 29), (2069, 29))

In [9]:
# [9] (C) X1(장비명) 누설방지 타겟 인코딩 + 빈도 (train-only foldwise)
def x1_per_serial(df):
    s=(df.sort_values(["Serial Number","TIMESTAMP"])
         .groupby("Serial Number")["X1"].apply(lambda x: x.dropna().iloc[0] if len(x.dropna()) else "UNK"))
    return s.astype(str)

x1_tr = x1_per_serial(train)
x1_te = x1_per_serial(test)
y_serial = train_y.set_index("Serial Number")["Y"].astype(int).reindex(x1_tr.index).fillna(0).astype(int)

def target_encode_foldwise(x1, y, x1_test, n_splits=N_SPLITS_2ND, m=80):
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    prior = float(y.mean())
    te_oof=np.zeros(len(x1), dtype=np.float32); freq_oof=np.zeros(len(x1), dtype=np.float32)
    x1=x1.reset_index(drop=True); y=y.reset_index(drop=True)

    for tr,va in cv.split(np.zeros(len(y)), y):
        x_tr=x1.iloc[tr]; y_tr=y.iloc[tr]
        stat=(pd.DataFrame({"x":x_tr,"y":y_tr}).groupby("x")["y"].agg(["sum","count"]))
        stat["te"]=(stat["sum"]+m*prior)/(stat["count"]+m)
        te_map=stat["te"].to_dict(); ct_map=stat["count"].to_dict()
        te_oof[va]=x1.iloc[va].map(te_map).fillna(prior).astype(np.float32)
        freq_oof[va]=np.log1p(x1.iloc[va].map(ct_map).fillna(0).astype(np.float32))

    stat_full=(pd.DataFrame({"x":x1,"y":y}).groupby("x")["y"].agg(["sum","count"]))
    stat_full["te"]=(stat_full["sum"]+m*prior)/(stat_full["count"]+m)
    te_map=stat_full["te"].to_dict(); ct_map=stat_full["count"].to_dict()
    te_test  = x1_test.map(te_map).fillna(prior).astype(np.float32).values
    freq_test= np.log1p(x1_test.map(ct_map).fillna(0).astype(np.float32)).values
    return te_oof, freq_oof, te_test, freq_test

x1_te_oof, x1_ct_oof, x1_te_test, x1_ct_test = target_encode_foldwise(x1_tr, y_serial, x1_te, n_splits=N_SPLITS_2ND, m=80)
X1_train = pd.DataFrame({"x1_te":x1_te_oof, "x1_logfreq":x1_ct_oof}, index=x1_tr.index)
X1_test  = pd.DataFrame({"x1_te":x1_te_test, "x1_logfreq":x1_ct_test}, index=x1_te.index)
X1_train.head()

,x1_te,x1_logfreq
Serial Number,,
19,0.146518,0.0
42,0.146518,0.0
48,0.146518,0.0
51,0.146518,0.0
67,0.146518,0.0


In [10]:
# [10] (A)+(B')+TE 결합 → 2단계 입력
Z_train_full = X_train_big.join([P_train, X1_train], how="left").fillna(0.0).astype("float32")
Z_test_full  = X_test_big .join([P_test , X1_test ], how="left").fillna(0.0).astype("float32")

# HGB 안정 위해 확률계열만 표준화 (fit=train, transform=test)
scale_cols = [c for c in Z_train_full.columns if (c.startswith("p_") or c in ("p_last","p_mean","p_max"))]
scaler = StandardScaler()
Z_train_full.loc[:, scale_cols] = scaler.fit_transform(Z_train_full[scale_cols])
Z_test_full.loc[:,  scale_cols] = scaler.transform(Z_test_full[scale_cols])

y = train_y.set_index("Serial Number")["Y"].astype(int).reindex(Z_train_full.index)
Z_train_full.shape, Z_test_full.shape, y.value_counts()

((8272, 882),
 (2069, 882),
 0    7060
 1    1212
 Name: Y, dtype: int64)

In [11]:
# [11] LGB 빠른 중요도 (train-only) → Top‑K 피처 선택
pos_ratio = float(y.mean()); neg_ratio = 1-pos_ratio
spw = neg_ratio/max(pos_ratio, 1e-6)
print(f"pos={pos_ratio:.4f}, spw≈{spw:.2f}")

quick_params = dict(
    objective="binary", metric="auc",
    learning_rate=0.05, num_leaves=96, feature_fraction=0.7,
    bagging_fraction=0.7, bagging_freq=1, max_bin=255,
    min_data_in_leaf=50, lambda_l2=1.0, seed=SEED, verbose=-1, scale_pos_weight=spw
)

feat_imp = pd.Series(0.0, index=Z_train_full.columns)
cv_rank = StratifiedKFold(n_splits=(3 if FAST_MODE else 5), shuffle=True, random_state=SEED)
for tr_idx,va_idx in cv_rank.split(Z_train_full, y):
    dtr = lgb.Dataset(Z_train_full.iloc[tr_idx], label=y.iloc[tr_idx])
    dva = lgb.Dataset(Z_train_full.iloc[va_idx], label=y.iloc[va_idx])
    m = lgb.train(quick_params, dtr, num_boost_round=(700 if FAST_MODE else 1500),
                  valid_sets=[dva], callbacks=[lgb.early_stopping(80 if FAST_MODE else 150, verbose=False)])
    feat_imp += pd.Series(m.feature_importance(importance_type="gain"), index=Z_train_full.columns)
    del m,dtr,dva; gc.collect()

feat_imp = feat_imp.sort_values(ascending=False)

TOPK_FINAL = 900   # FAST_MODE=False → 넉넉히 사용
keep_cols  = feat_imp.index[:TOPK_FINAL]
Xtr = Z_train_full[keep_cols].copy().astype("float32")
Xte = Z_test_full [keep_cols].copy().astype("float32")
print("Selected features:", len(keep_cols))

pos=0.1465, spw≈5.83
Selected features: 882


In [12]:
# [12] 2단계 모델 파라미터 (LGB/XGB/HGB)
params_lgb = dict(
    objective="binary", metric="auc",
    learning_rate=0.02, num_leaves=192, max_depth=-1,
    min_data_in_leaf=80, min_sum_hessian_in_leaf=1e-2,
    feature_fraction=0.7, bagging_fraction=0.7, bagging_freq=1,
    max_bin=511, lambda_l2=2.0, extra_trees=True, path_smooth=20,
    seed=SEED, verbose=-1, scale_pos_weight=spw,
)

params_xgb = dict(
    objective="binary:logistic", eval_metric="aucpr",
    tree_method="hist", grow_policy="lossguide",
    max_leaves=416, max_depth=0, max_bin=512,
    learning_rate=0.03, min_child_weight=12, subsample=0.8, colsample_bytree=0.7,
    reg_lambda=2.0, reg_alpha=0.0, max_delta_step=1, scale_pos_weight=spw, random_state=SEED
)

hgb_params = dict(
    loss="log_loss", max_iter=HGB2_ITERS,
    learning_rate=0.04,
    max_leaf_nodes=127,
    min_samples_leaf=20, l2_regularization=0.0,
    early_stopping=True, validation_fraction=0.1, n_iter_no_change=50,
    random_state=SEED
)

ROUNDS_FULL = {"lgb": 3800, "xgb": 3300, "hgb": HGB2_ITERS}
ES_FULL     = {"lgb": 200,  "xgb": 200}

In [13]:
# [13] 학습 루틴 (OOF/TE 반환)
def train_lgb(X, y, Xtest, params, num_rounds, es_rounds, desc="[LGB]"):
    Xn=X.values; Xten=Xtest.values
    oof=np.zeros(len(Xn), dtype=np.float32); tef=np.zeros((len(Xten), N_SPLITS_2ND), dtype=np.float32)
    cv=StratifiedKFold(n_splits=N_SPLITS_2ND, shuffle=True, random_state=SEED)
    p=params.copy(); p.update(dict(first_metric_only=True))
    pbar=_tqdm(total=N_SPLITS_2ND, desc=desc, leave=False); k=0
    for tr,va in cv.split(Xn, y.values):
        dtr=lgb.Dataset(Xn[tr], label=y.values[tr]); dva=lgb.Dataset(Xn[va], label=y.values[va], reference=dtr)
        m=lgb.train(p, dtr, num_boost_round=num_rounds, valid_sets=[dva],
                    callbacks=[lgb.early_stopping(es_rounds, verbose=False)])
        oof[va]=m.predict(Xn[va], num_iteration=m.best_iteration).astype(np.float32)
        tef[:,k]=m.predict(Xten, num_iteration=m.best_iteration).astype(np.float32)
        k+=1; pbar.update(1); del m,dtr,dva; gc.collect()
    pbar.close(); return oof, tef.mean(axis=1)

def train_xgb(X, y, Xtest, params, num_rounds, es_rounds, desc="[XGB]"):
    Xn=X.values; Xten=Xtest.values
    oof=np.zeros(len(Xn), dtype=np.float32); tef=np.zeros((len(Xten), N_SPLITS_2ND), dtype=np.float32)
    cv=StratifiedKFold(n_splits=N_SPLITS_2ND, shuffle=True, random_state=SEED)
    p=params.copy(); p.setdefault("nthread",-1)
    pbar=_tqdm(total=N_SPLITS_2ND, desc=desc, leave=False); k=0
    dte=xgb.DMatrix(Xten)
    for tr,va in cv.split(Xn, y.values):
        dtr=xgb.DMatrix(Xn[tr], label=y.values[tr]); dva=xgb.DMatrix(Xn[va], label=y.values[va])
        m=xgb.train(p, dtr, num_boost_round=num_rounds, evals=[(dva,"val")],
                    early_stopping_rounds=es_rounds, verbose_eval=False)
        oof[va]=m.predict(xgb.DMatrix(Xn[va]), iteration_range=(0, m.best_iteration)).astype(np.float32)
        tef[:,k]=m.predict(dte, iteration_range=(0, m.best_iteration)).astype(np.float32)
        k+=1; pbar.update(1); del m,dtr,dva; gc.collect()
    pbar.close(); return oof, tef.mean(axis=1)

def train_hgb(X, y, Xtest, params, max_iter, desc="[HGB]"):
    p=params.copy(); p["max_iter"]=max_iter
    oof=np.zeros(len(X), dtype=np.float32); tef=np.zeros((len(Xtest), N_SPLITS_2ND), dtype=np.float32)
    cv=StratifiedKFold(n_splits=N_SPLITS_2ND, shuffle=True, random_state=SEED)
    pbar=_tqdm(total=N_SPLITS_2ND, desc=desc, leave=False); k=0
    for tr,va in cv.split(X.values, y.values):
        clf=HistGradientBoostingClassifier(**p)
        clf.fit(X.iloc[tr], y.iloc[tr])
        oof[va]=clf.predict_proba(X.iloc[va])[:,1].astype(np.float32)
        tef[:,k]=clf.predict_proba(Xtest)[:,1].astype(np.float32)
        k+=1; pbar.update(1); del clf; gc.collect()
    pbar.close(); return oof, tef.mean(axis=1)

In [14]:
# [14] 학습 실행 (단일모델 OOF F1(macro))
oof_lgb , te_lgb  = train_lgb(Xtr, y, Xte, params_lgb, num_rounds=ROUNDS_FULL["lgb"], es_rounds=ES_FULL["lgb"], desc="[LGB]")
oof_xgb , te_xgb  = train_xgb(Xtr, y, Xte, params_xgb, num_rounds=ROUNDS_FULL["xgb"], es_rounds=ES_FULL["xgb"], desc="[XGB]")
oof_hgb , te_hgb  = train_hgb(Xtr, y, Xte, hgb_params, max_iter=ROUNDS_FULL["hgb"], desc="[HGB]")

for name, o in [("LGB", oof_lgb), ("XGB", oof_xgb), ("HGB", oof_hgb)]:
    f1, thr = _best_f1_thr(y.values, o, average="macro")
    print(f"[{name}] OOF F1(macro)={f1:.5f} @ thr≈{thr:.3f}")

[LGB] OOF F1(macro)=0.98515 @ thr≈0.505
[XGB] OOF F1(macro)=0.98639 @ thr≈0.905
[HGB] OOF F1(macro)=0.98815 @ thr≈0.645


In [15]:
# [15] (선택) Isotonic 보정 — OOF로만 학습, test는 변환만
def calibrate_opt(oof, y, test, use_iso=CALIB_ISO_2ND):
    if not use_iso: return oof, test
    ir = IsotonicRegression(out_of_bounds="clip")
    ir.fit(oof, y)
    return ir.transform(oof), ir.transform(test)

oof_lgb, te_lgb = calibrate_opt(oof_lgb, y.values, te_lgb, CALIB_ISO_2ND)
oof_xgb, te_xgb = calibrate_opt(oof_xgb, y.values, te_xgb, CALIB_ISO_2ND)
oof_hgb, te_hgb = calibrate_opt(oof_hgb, y.values, te_hgb, CALIB_ISO_2ND)

In [16]:
# [16] 블렌딩 (prob/logit 모두 OOF로 탐색 → 더 좋은 쪽 선택), 임계값 후보들(OOB 통계만)
cand = {"lgb": (oof_lgb, te_lgb),
        "xgb": (oof_xgb, te_xgb),
        "hgb": (oof_hgb, te_hgb)}
keys = list(cand.keys())

def blend_prob(W, use_rank=False):
    oofs=[]; tests=[]
    for k,w in W.items():
        o,t=cand[k]
        _o = _rank_norm(o) if use_rank else o
        _t = _rank_norm(t) if use_rank else t
        oofs.append(_o*w); tests.append(_t*w)
    return np.sum(oofs,axis=0), np.sum(tests,axis=0)

def blend_logit(W, use_rank=False):
    oofs=[]; tests=[]
    for k,w in W.items():
        o,t=cand[k]
        _o = _rank_norm(o) if use_rank else o
        _t = _rank_norm(t) if use_rank else t
        oofs.append(_logit(_o)*w); tests.append(_logit(_t)*w)
    return _ilogit(np.sum(oofs,axis=0)), _ilogit(np.sum(tests,axis=0))

# 1) coarse 탐색 (0.1 step)
grid = np.arange(0.2, 0.9, 0.1)
best_prob=(0.0,None,None); best_logit=(0.0,None,None)
for a in grid:
    for b in grid:
        c = 1.0 - a - b
        if c<=0: continue
        W = {keys[0]:a, keys[1]:b, keys[2]:c}
        o,_t = blend_prob(W, use_rank=False)
        f1,_  = _best_f1_thr(y.values, o, average="macro")
        if f1>best_prob[0]: best_prob=(f1,W,_t)
        o,_t = blend_logit(W, use_rank=False)
        f1,_  = _best_f1_thr(y.values, o, average="macro")
        if f1>best_logit[0]: best_logit=(f1,W,_t)

# 2) 공간 선택 (OOF 성능으로만)
space, best = ("prob", best_prob) if best_prob[0] >= best_logit[0] else ("logit", best_logit)
W = best[1]

# 3) fine 탐색 (0.02 step) — 선택 공간 내에서만, 주변 샘플링
fine = np.arange(-0.10, 0.101, 0.02)  # 주변 perturbation
def normalize_w(a,b,c):
    s=a+b+c
    if s<=0: return None
    return a/s, b/s, c/s

def try_fine(space, base_W):
    ks = list(base_W.keys())
    base = np.array([base_W[ks[0]], base_W[ks[1]], base_W[ks[2]]], dtype=float)
    best_local = (0.0, base_W, None)
    for da in fine:
        for db in fine:
            a = base[0]+da; b=base[1]+db; c=base[2]-(da+db)
            norm = normalize_w(a,b,c)
            if norm is None or min(norm)<0: continue
            Wt = {ks[0]:norm[0], ks[1]:norm[1], ks[2]:norm[2]}
            if space=="logit":
                o,_t = blend_logit(Wt, use_rank=False)
            else:
                o,_t = blend_prob(Wt, use_rank=False)
            f1,_ = _best_f1_thr(y.values, o, average="macro")
            if f1>best_local[0]: best_local=(f1, Wt, _t)
    return best_local

best = try_fine(space, W)
W = best[1]
bl_oof, bl_test = (blend_logit(W, use_rank=False) if space=="logit" else blend_prob(W, use_rank=False))

# 4) 임계값 후보 (OOF/훈련 통계만)
f1_oof,  thr_oof  = _best_f1_thr(y.values, bl_oof, average="macro")

thr_list=[]
cv_thr = StratifiedKFold(n_splits=N_SPLITS_2ND, shuffle=True, random_state=SEED)
for _, va in cv_thr.split(bl_oof, y.values):
    _, thr_f = _best_f1_thr(y.values[va], bl_oof[va], average="macro")
    thr_list.append(thr_f)
thr_fold = float(np.median(thr_list))

target_rate = float(train_y["Y"].mean()) * RATE_BUF
thr_rate = float(np.quantile(bl_oof, 1.0 - target_rate))

# 후보 중 OOF F1이 가장 높은 것을 선택
thr_cands = {"fold":thr_fold, "oof":thr_oof, "rate":thr_rate}
scores = {name: float(f1_score(y.values, (bl_oof>thr).astype(int), average="macro"))
          for name, thr in thr_cands.items()}
thr_name = max(scores.items(), key=lambda x: x[1])[0]
final_thr = thr_cands[thr_name]

final_oof  = bl_oof
final_test = bl_test
final_f1   = f1_score(y.values, (final_oof>final_thr).astype(int), average="macro")

print(f"[BLEND] space={space} W={W}")
print(f"  thr(oof)={thr_oof:.4f}, thr(fold)={thr_fold:.4f}, thr(rate@{target_rate:.4f})={thr_rate:.4f}")
print(f"  OOF F1@oof={scores['oof']:.5f} | @fold={scores['fold']:.5f} | @rate={scores['rate']:.5f}")
print(f"→ choose_thr={thr_name} (thr={final_thr:.4f}) | OOF F1(macro)={final_f1:.5f}")

[BLEND] space=prob W={'lgb': 0.1, 'xgb': 0.1, 'hgb': 0.8}
  thr(oof)=0.7300, thr(fold)=0.5900, thr(rate@0.1509)=0.1930
  OOF F1@oof=0.98788 | @fold=0.98747 | @rate=0.98592
→ choose_thr=oof (thr=0.7300) | OOF F1(macro)=0.98788


In [17]:
# [17] 메타 스태킹(LogReg, balanced) — 더 좋으면 교체 (OOF 기반 결정)
stack_tr = np.vstack([cand[k][0] for k in keys]).T
stack_te = np.vstack([cand[k][1] for k in keys]).T

meta = LogisticRegression(C=1.0, solver="liblinear", class_weight="balanced", random_state=SEED)
meta.fit(stack_tr, y.values)
oof_meta  = meta.predict_proba(stack_tr)[:,1]
test_meta = meta.predict_proba(stack_te)[:,1]
f1_m, thr_m = _best_f1_thr(y.values, oof_meta, average="macro")
print(f"[META] OOF F1(macro)={f1_m:.5f} @ thr≈{thr_m:.3f}")

if f1_m >= final_f1:
    final_oof, final_test, final_thr = oof_meta, test_meta, thr_m
    print("→ Use META")
else:
    print("→ Use BLEND(thr as chosen)")

[META] OOF F1(macro)=0.98673 @ thr≈0.885
→ Use BLEND(thr as chosen)


In [18]:
# [18] 제출 저장
sub = pd.read_csv(f"{DATA_DIR}/test_y.csv", index_col="Serial Number")
sub["Y"] = (final_test > final_thr).astype(int)  # 숫자 0/1
sub.to_csv("submission.csv", index_label="Serial Number")

print(sub["Y"].value_counts(dropna=False))
print("submission.csv saved.")
print("Pred 1 ratio:", float(sub["Y"].mean()))
print("Train 1 ratio:", float(train_y["Y"].mean()))

0    1746
1     323
Name: Y, dtype: int64
submission.csv saved.
Pred 1 ratio: 0.15611406476558723
Train 1 ratio: 0.1465183752417795
